# Multimodal Sensory Dataset Generation

This notebook generates the eight-dimensional sensory dataset described in Section 4.2 of the paper.

**Dataset Specifications:**
- 8 sensory modalities (visual, auditory, tactile, olfactory, gustatory, proprioceptive, vestibular, interoceptive)
- Total dimensionality: 359 features per timestep
- Temporal structure: 60 seconds at 1Hz sampling (60 timesteps)
- 5,000 training examples, 1,000 validation, 1,000 test examples
- Diverse scenarios: walking, running, sitting, eating, drinking, listening, viewing

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Create output directories
Path('data').mkdir(exist_ok=True)
Path('data/train').mkdir(exist_ok=True)
Path('data/val').mkdir(exist_ok=True)
Path('data/test').mkdir(exist_ok=True)

## 1. Define Sensory Modality Classes

In [ ]:
class SensoryModality:
    """Base class for sensory modalities"""
    def __init__(self, name: str, dimensions: int):
        self.name = name
        self.dimensions = dimensions
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        """Generate sensory data for given scenario"""
        raise NotImplementedError

class VisualModality(SensoryModality):
    """Visual sensation: EM spectrum 300-1000nm, 5nm intervals = 140 dimensions"""
    def __init__(self):
        super().__init__('visual', 140)
        self.wavelengths = np.arange(300, 1005, 5)  # 300-1000nm, 5nm steps
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'viewing':
            # Rich visual input across spectrum
            base_spectrum = np.random.rand(self.dimensions) * 0.8
            # Add temporal variation
            for t in range(timesteps):
                data[t] = base_spectrum + np.random.randn(self.dimensions) * 0.1
        
        elif scenario in ['walking', 'running']:
            # Moderate visual input with motion
            base_spectrum = np.random.rand(self.dimensions) * 0.5
            for t in range(timesteps):
                # Simulate visual motion blur
                motion_factor = np.sin(t * 0.1) * 0.2
                data[t] = base_spectrum + motion_factor + np.random.randn(self.dimensions) * 0.05
        
        else:  # sitting, eating, drinking, listening
            # Lower visual variation
            base_spectrum = np.random.rand(self.dimensions) * 0.3
            for t in range(timesteps):
                data[t] = base_spectrum + np.random.randn(self.dimensions) * 0.02
        
        return np.clip(data, 0, 1)

class AuditoryModality(SensoryModality):
    """Auditory sensation: 1Hz-100kHz, 128 FFT bins"""
    def __init__(self):
        super().__init__('auditory', 128)
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'listening':
            # Rich auditory input
            for t in range(timesteps):
                # Simulate speech/music with harmonic structure
                fundamental = np.random.randint(5, 20)
                for harmonic in range(1, 6):
                    freq_idx = min(fundamental * harmonic, self.dimensions - 1)
                    data[t, freq_idx] = np.random.rand() * 0.8
        
        elif scenario in ['walking', 'running']:
            # Rhythmic footstep sounds
            step_freq = 2 if scenario == 'walking' else 3
            for t in range(timesteps):
                if t % step_freq == 0:
                    data[t, :30] = np.random.rand(30) * 0.5  # Low frequency impact
        
        else:  # sitting, eating, drinking, viewing
            # Ambient low-level sounds
            for t in range(timesteps):
                data[t] = np.random.rand(self.dimensions) * 0.1
        
        return np.clip(data, 0, 1)

class TactileModality(SensoryModality):
    """Tactile sensation: mechanical vibrations 5-800Hz, 32 dimensions"""
    def __init__(self):
        super().__init__('tactile', 32)
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario in ['walking', 'running']:
            # Strong rhythmic tactile input from ground contact
            step_freq = 2 if scenario == 'walking' else 3
            for t in range(timesteps):
                if t % step_freq == 0:
                    data[t] = np.random.rand(self.dimensions) * 0.7
        
        elif scenario in ['eating', 'drinking']:
            # Oral tactile sensations
            for t in range(timesteps):
                data[t, :10] = np.random.rand(10) * 0.5
        
        else:  # sitting, listening, viewing
            # Minimal tactile input
            for t in range(timesteps):
                data[t] = np.random.rand(self.dimensions) * 0.1
        
        return np.clip(data, 0, 1)

class OlfactoryModality(SensoryModality):
    """Olfactory sensation: 10 categories, 10 dimensions"""
    def __init__(self):
        super().__init__('olfactory', 10)
        self.categories = ['fruity', 'floral', 'musky', 'woody', 'earthy', 
                          'spicy', 'chemical', 'pungent', 'ethereal', 'minty']
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'eating':
            # Food-related smells
            active_categories = [0, 1, 5]  # fruity, floral, spicy
            for t in range(timesteps):
                for cat in active_categories:
                    data[t, cat] = np.random.rand() * 0.7
        
        elif scenario == 'drinking':
            # Beverage-related smells
            active_categories = [0, 9]  # fruity, minty
            for t in range(timesteps):
                for cat in active_categories:
                    data[t, cat] = np.random.rand() * 0.6
        
        else:
            # Background ambient smells
            for t in range(timesteps):
                data[t] = np.random.rand(self.dimensions) * 0.2
        
        return np.clip(data, 0, 1)

class GustatoryModality(SensoryModality):
    """Gustatory sensation: 5 tastes, 5 dimensions"""
    def __init__(self):
        super().__init__('gustatory', 5)
        self.tastes = ['sweet', 'sour', 'salty', 'bitter', 'umami']
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'eating':
            # Complex taste profile
            taste_profile = np.random.rand(self.dimensions) * 0.8
            for t in range(timesteps):
                # Taste persists with slight variation
                data[t] = taste_profile + np.random.randn(self.dimensions) * 0.1
        
        elif scenario == 'drinking':
            # Simpler taste profile
            taste_profile = np.random.rand(self.dimensions) * 0.5
            for t in range(timesteps):
                data[t] = taste_profile + np.random.randn(self.dimensions) * 0.05
        
        else:
            # No taste
            data = np.zeros((timesteps, self.dimensions))
        
        return np.clip(data, 0, 1)

class ProprioceptiveModality(SensoryModality):
    """Proprioceptive sensation: 20 joint angles + 10 muscle tensions, 30 dimensions"""
    def __init__(self):
        super().__init__('proprioceptive', 30)
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'walking':
            # Rhythmic joint movements
            for t in range(timesteps):
                # Simulate gait cycle
                phase = (t % 20) / 20.0 * 2 * np.pi
                data[t, :20] = 0.5 + 0.3 * np.sin(phase + np.arange(20) * 0.1)
                data[t, 20:] = 0.3 + 0.2 * np.cos(phase)
        
        elif scenario == 'running':
            # More dynamic joint movements
            for t in range(timesteps):
                phase = (t % 15) / 15.0 * 2 * np.pi
                data[t, :20] = 0.5 + 0.5 * np.sin(phase + np.arange(20) * 0.15)
                data[t, 20:] = 0.5 + 0.3 * np.cos(phase)
        
        elif scenario == 'sitting':
            # Static posture with small variations
            sitting_posture = np.random.rand(self.dimensions) * 0.3
            for t in range(timesteps):
                data[t] = sitting_posture + np.random.randn(self.dimensions) * 0.02
        
        else:  # eating, drinking, listening, viewing
            # Moderate joint activity
            for t in range(timesteps):
                data[t] = 0.3 + np.random.rand(self.dimensions) * 0.2
        
        return np.clip(data, 0, 1)

class VestibularModality(SensoryModality):
    """Vestibular sensation: 3-axis acceleration + 3-axis angular velocity, 6 dimensions"""
    def __init__(self):
        super().__init__('vestibular', 6)
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        if scenario == 'walking':
            # Rhythmic head movement
            for t in range(timesteps):
                phase = (t % 20) / 20.0 * 2 * np.pi
                data[t, :3] = [0.5, 0.1 * np.sin(phase), 0.05 * np.cos(phase)]  # acceleration
                data[t, 3:] = [0.02 * np.sin(phase), 0.01, 0.01 * np.cos(phase)]  # angular velocity
        
        elif scenario == 'running':
            # More dynamic head movement
            for t in range(timesteps):
                phase = (t % 15) / 15.0 * 2 * np.pi
                data[t, :3] = [0.7, 0.2 * np.sin(phase), 0.1 * np.cos(phase)]
                data[t, 3:] = [0.05 * np.sin(phase), 0.02, 0.02 * np.cos(phase)]
        
        elif scenario == 'sitting':
            # Minimal movement
            for t in range(timesteps):
                data[t] = np.random.randn(self.dimensions) * 0.01
        
        else:  # eating, drinking, listening, viewing
            # Small head movements
            for t in range(timesteps):
                data[t] = np.random.randn(self.dimensions) * 0.05
        
        return np.clip(data, -1, 1)

class InteroceptiveModality(SensoryModality):
    """Interoceptive sensation: HR, respiration, skin conductance, temp, hunger, thirst, fatigue, pain, 8 dimensions"""
    def __init__(self):
        super().__init__('interoceptive', 8)
        self.signals = ['heart_rate', 'respiration', 'skin_conductance', 'temperature', 
                       'hunger', 'thirst', 'fatigue', 'pain']
    
    def generate(self, scenario: str, timesteps: int) -> np.ndarray:
        data = np.zeros((timesteps, self.dimensions))
        
        # Base physiological state
        base_state = np.array([0.5, 0.4, 0.3, 0.5, 0.3, 0.3, 0.2, 0.1])  # normalized values
        
        if scenario == 'running':
            # Elevated heart rate and respiration
            for t in range(timesteps):
                hr_variation = 0.3 + 0.1 * np.sin(t * 0.3)
                data[t] = base_state + np.array([hr_variation, 0.3, 0.2, 0.1, 0, 0.1, 0.1, 0.05])
        
        elif scenario == 'walking':
            # Moderately elevated
            for t in range(timesteps):
                hr_variation = 0.15 + 0.05 * np.sin(t * 0.2)
                data[t] = base_state + np.array([hr_variation, 0.15, 0.1, 0.05, 0, 0.05, 0.05, 0.02])
        
        elif scenario == 'eating':
            # Changing hunger signal
            for t in range(timesteps):
                hunger_change = -0.3 * (t / timesteps)  # Decreasing hunger
                data[t] = base_state + np.array([0, 0, 0, 0, hunger_change, 0, -0.05, 0])
        
        elif scenario == 'drinking':
            # Changing thirst signal
            for t in range(timesteps):
                thirst_change = -0.3 * (t / timesteps)
                data[t] = base_state + np.array([0, 0, 0, 0, 0, thirst_change, 0, 0])
        
        else:  # sitting, listening, viewing
            # Resting state
            for t in range(timesteps):
                data[t] = base_state + np.random.randn(self.dimensions) * 0.02
        
        return np.clip(data, 0, 1)

## 2. Dataset Generator Class

In [ ]:
class MultimodalSensoryDataset:
    """Generator for complete multimodal sensory dataset"""
    
    def __init__(self, timesteps: int = 60):
        self.timesteps = timesteps
        self.scenarios = ['walking', 'running', 'sitting', 'eating', 'drinking', 'listening', 'viewing']
        
        # Initialize all modalities
        self.modalities = [
            VisualModality(),
            AuditoryModality(),
            TactileModality(),
            OlfactoryModality(),
            GustatoryModality(),
            ProprioceptiveModality(),
            VestibularModality(),
            InteroceptiveModality()
        ]
        
        self.total_dimensions = sum(m.dimensions for m in self.modalities)
        print(f"Total dimensions per timestep: {self.total_dimensions}")
        print(f"Total data points per example: {self.total_dimensions * self.timesteps}")
    
    def generate_example(self, scenario: str) -> Dict:
        """Generate a single example for given scenario"""
        example = {
            'scenario': scenario,
            'timesteps': self.timesteps,
            'modalities': {}
        }
        
        # Generate data for each modality
        for modality in self.modalities:
            data = modality.generate(scenario, self.timesteps)
            example['modalities'][modality.name] = data
        
        return example
    
    def generate_dataset(self, n_examples: int, balanced: bool = True) -> List[Dict]:
        """Generate dataset with n_examples"""
        dataset = []
        
        if balanced:
            # Balance examples across scenarios
            n_per_scenario = n_examples // len(self.scenarios)
            remainder = n_examples % len(self.scenarios)
            
            for i, scenario in enumerate(self.scenarios):
                n_scenario = n_per_scenario + (1 if i < remainder else 0)
                for _ in range(n_scenario):
                    example = self.generate_example(scenario)
                    dataset.append(example)
        else:
            # Random scenario selection
            for _ in range(n_examples):
                scenario = np.random.choice(self.scenarios)
                example = self.generate_example(scenario)
                dataset.append(example)
        
        return dataset
    
    def save_dataset(self, dataset: List[Dict], split: str):
        """Save dataset to disk"""
        output_dir = Path(f'data/{split}')
        
        for idx, example in enumerate(dataset):
            # Convert to serializable format
            serializable = {
                'scenario': example['scenario'],
                'timesteps': example['timesteps'],
                'modalities': {}
            }
            
            for modality_name, data in example['modalities'].items():
                serializable['modalities'][modality_name] = data.tolist()
            
            # Save as JSON
            with open(output_dir / f'example_{idx:05d}.json', 'w') as f:
                json.dump(serializable, f)
        
        print(f"Saved {len(dataset)} examples to {output_dir}")
    
    def create_ontology(self) -> Dict:
        """Create machine-readable ontology"""
        ontology = {
            'modalities': [],
            'total_dimensions': self.total_dimensions,
            'temporal_structure': {
                'duration_seconds': self.timesteps,
                'sampling_rate_hz': 1
            }
        }
        
        for modality in self.modalities:
            modality_info = {
                'name': modality.name,
                'dimensions': modality.dimensions,
                'description': f'{modality.__class__.__name__} sensory information'
            }
            
            # Add specific details
            if isinstance(modality, VisualModality):
                modality_info['physical_basis'] = 'Electromagnetic radiation'
                modality_info['range'] = '300-1000nm'
                modality_info['sampling'] = '5nm intervals'
            elif isinstance(modality, AuditoryModality):
                modality_info['physical_basis'] = 'Mechanical pressure waves'
                modality_info['range'] = '1Hz-100kHz'
                modality_info['representation'] = '128 FFT bins'
            elif isinstance(modality, TactileModality):
                modality_info['physical_basis'] = 'Mechanical vibrations'
                modality_info['range'] = '5-800Hz'
            elif isinstance(modality, OlfactoryModality):
                modality_info['physical_basis'] = 'Chemical molecules'
                modality_info['categories'] = modality.categories
            elif isinstance(modality, GustatoryModality):
                modality_info['physical_basis'] = 'Chemical molecules'
                modality_info['categories'] = modality.tastes
            elif isinstance(modality, ProprioceptiveModality):
                modality_info['physical_basis'] = 'Body position and movement'
                modality_info['components'] = '20 joint angles, 10 muscle tensions'
            elif isinstance(modality, VestibularModality):
                modality_info['physical_basis'] = 'Head acceleration and rotation'
                modality_info['components'] = '3-axis acceleration, 3-axis angular velocity'
            elif isinstance(modality, InteroceptiveModality):
                modality_info['physical_basis'] = 'Internal physiological state'
                modality_info['signals'] = modality.signals
            
            ontology['modalities'].append(modality_info)
        
        return ontology

## 3. Generate Complete Dataset

In [ ]:
# Initialize dataset generator
generator = MultimodalSensoryDataset(timesteps=60)

# Generate training set (5000 examples)
print("Generating training set...")
train_dataset = generator.generate_dataset(5000, balanced=True)
generator.save_dataset(train_dataset, 'train')

# Generate validation set (1000 examples)
print("\nGenerating validation set...")
val_dataset = generator.generate_dataset(1000, balanced=True)
generator.save_dataset(val_dataset, 'val')

# Generate test set (1000 examples)
print("\nGenerating test set...")
test_dataset = generator.generate_dataset(1000, balanced=True)
generator.save_dataset(test_dataset, 'test')

print("\nDataset generation complete!")

## 4. Create and Save Ontology

In [ ]:
# Create ontology
ontology = generator.create_ontology()

# Save ontology
with open('data/ontology.json', 'w') as f:
    json.dump(ontology, f, indent=2)

print("Ontology saved to data/ontology.json")
print("\nOntology summary:")
print(f"Total modalities: {len(ontology['modalities'])}")
print(f"Total dimensions: {ontology['total_dimensions']}")
print("\nModalities:")
for mod in ontology['modalities']:
    print(f"  - {mod['name']}: {mod['dimensions']} dimensions")

## 5. Visualize Sample Data

In [ ]:
# Load a sample example
with open('data/train/example_00000.json', 'r') as f:
    sample = json.load(f)

# Create visualization
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
fig.suptitle(f'Sample Multimodal Sensory Data - Scenario: {sample["scenario"]}', fontsize=16)

modality_names = list(sample['modalities'].keys())

for idx, (ax, modality_name) in enumerate(zip(axes.flatten(), modality_names)):
    data = np.array(sample['modalities'][modality_name])
    
    if data.shape[1] > 20:  # High-dimensional modalities
        # Show as heatmap
        im = ax.imshow(data.T, aspect='auto', cmap='viridis')
        ax.set_ylabel('Feature dimension')
        plt.colorbar(im, ax=ax)
    else:  # Low-dimensional modalities
        # Show as line plots
        for dim in range(min(data.shape[1], 10)):
            ax.plot(data[:, dim], alpha=0.7, label=f'Dim {dim}')
        if data.shape[1] <= 10:
            ax.legend(fontsize=8)
        ax.set_ylabel('Value')
    
    ax.set_xlabel('Timestep')
    ax.set_title(f'{modality_name.capitalize()} ({data.shape[1]} dims)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization saved to data/sample_visualization.png")

## 6. Dataset Statistics

In [ ]:
# Compute statistics across training set
print("Computing dataset statistics...")

scenario_counts = {scenario: 0 for scenario in generator.scenarios}
modality_stats = {mod.name: {'mean': [], 'std': []} for mod in generator.modalities}

for example in train_dataset[:100]:  # Sample for efficiency
    scenario_counts[example['scenario']] += 1
    
    for modality_name, data in example['modalities'].items():
        modality_stats[modality_name]['mean'].append(np.mean(data))
        modality_stats[modality_name]['std'].append(np.std(data))

# Print statistics
print("\nScenario distribution (sample):")
for scenario, count in scenario_counts.items():
    print(f"  {scenario}: {count}")

print("\nModality statistics (across sample):")
for modality_name, stats in modality_stats.items():
    print(f"  {modality_name}:")
    print(f"    Mean: {np.mean(stats['mean']):.4f} (±{np.std(stats['mean']):.4f})")
    print(f"    Std:  {np.mean(stats['std']):.4f} (±{np.std(stats['std']):.4f})")

## 7. Create Summary Report

In [ ]:
summary = {
    'dataset_name': 'Multimodal Sensory Dataset',
    'creation_date': pd.Timestamp.now().isoformat(),
    'splits': {
        'train': 5000,
        'validation': 1000,
        'test': 1000
    },
    'total_examples': 7000,
    'scenarios': generator.scenarios,
    'temporal_structure': {
        'duration_seconds': 60,
        'sampling_rate_hz': 1,
        'timesteps': 60
    },
    'dimensionality': {
        'per_timestep': generator.total_dimensions,
        'per_example': generator.total_dimensions * 60,
        'total_dataset': generator.total_dimensions * 60 * 7000
    },
    'modalities': [{
        'name': mod.name,
        'dimensions': mod.dimensions
    } for mod in generator.modalities]
}

with open('data/dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Dataset summary saved to data/dataset_summary.json")
print("\n=== DATASET GENERATION COMPLETE ===")
print(f"Total examples: {summary['total_examples']}")
print(f"Dimensions per timestep: {summary['dimensionality']['per_timestep']}")
print(f"Data points per example: {summary['dimensionality']['per_example']}")
print(f"Total data points: {summary['dimensionality']['total_dataset']:,}")